In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = '/content/drive/MyDrive/online+retail+ii/online_retail_II.xlsx'
df = pd.read_excel(file_path)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


# **Data Quality Check & Cleaning**

In [ ]:
df.shape

(525461, 8)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [ ]:
# percentage of missing customer IDs
null_values = df.isna().sum()
perct_null = (null_values / len(df)) * 100

missing_info = pd.DataFrame({
    'Missing Count': null_values,
    'Percentage Missing': perct_null.round(1)
})

print("Missing values and their percentages for Customer ID and Description:")
display(missing_info.loc[['Customer ID', 'Description']].sort_values(by='Percentage Missing', ascending=False))

Missing values and their percentages for Customer ID and Description:


,Missing Count,Percentage Missing
Customer ID,107927,20.5
Description,2928,0.6


In [ ]:
df.duplicated().sum()

np.int64(6865)

- There are **6.8k** duplicates.

- First I did sanity check to know which ones they are before dropping them because I have many **missing Customer IDs and product description**.

### Inspecting Duplicate Rows

In [ ]:
# Get all duplicate rows
duplicate_rows = df[df.duplicated(keep=False)]

# Sort by Invoice and StockCode to group similar entries
duplicate_rows = duplicate_rows.sort_values(by=['Invoice', 'StockCode', 'InvoiceDate'])

print(f"Total number of rows identified as duplicates: {len(duplicate_rows)}")

# Display the first few duplicate rows
display(duplicate_rows.head(20))

Total number of rows identified as duplicates: 13283


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


After reviewing the sample of duplicate rows, it looks like many of them are exact duplicates. For example, **rows 379 and 391** *both show Invoice 489517 for StockCode 21491, with all other details (Description, Quantity, Price, Customer ID, InvoiceDate, Country) being precisely the same.*

This means that for the same invoice, the exact same item, quantity, and price are recorded multiple times. In a retail dataset, this is usually an indication of redundant data entry rather than multiple distinct purchases or items within the same transaction. Although sometimes 'duplicates' might mean something else (like different items on the same invoice, or separate purchases), in this case, the rows are literally identical copies.

Therefore, I dropped the duplicates.

In [ ]:
# Drop exact duplicate rows
df.drop_duplicates(inplace=True)

print("DataFrame shape after dropping exact duplicates:")
print(df.shape)

DataFrame shape after dropping exact duplicates:
(518596, 8)


In [ ]:
# recheck missing values after dropping duplicates
null_values = df.isna().sum()
perct_null = (null_values / len(df)) * 100

missing_info = pd.DataFrame({
    'Missing Count': null_values,
    'Percentage Missing': perct_null.round(1)
})

print("Missing values and their percentages for Customer ID and Description:")
display(missing_info.loc[['Customer ID', 'Description']].sort_values(by='Percentage Missing', ascending=False))

Missing values and their percentages for Customer ID and Description:


,Missing Count,Percentage Missing
Customer ID,107833,20.8
Description,2928,0.6


### **Exploring Rows with Missing 'Customer ID'**

Next I examined the transactions where the `Customer ID` is missing. This might reveal patterns or provide insights into why these IDs are not recorded.

In [ ]:
missing_customer_id_df = df[df['Customer ID'].isna()]

print(f"Total number of rows with missing Customer ID: {len(missing_customer_id_df)}")

display(missing_customer_id_df.head(10))

Total number of rows with missing Customer ID: 107833


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom


The analysis of rows with missing 'Customer ID' reveals some interesting patterns:

**Total Count:** As previously identified, there are **107,833 transactions** where the 'Customer ID' is missing.

**Negative Quantities & Zero Prices:** Many of the transactions with missing 'Customer ID' also show negative Quantity values and Price values of 0.00 (e.g., rows 263, 283, 284, 470). *This often indicates returns, cancellations*, or possibly adjustments in the system that were not linked to a specific customer ID or did not involve a monetary transaction.

**Missing Description:** Some rows with missing 'Customer ID' also have missing 'Description' (e.g., row 470), suggesting multiple data quality issues co-occurring.

**Normal Transactions:** Interestingly, some rows (e.g., row 577) appear to be regular sales (positive quantity, non-zero price) but still lack a 'Customer ID'. *This could indicate guest checkouts or errors in customer data capture.*


*The missing 'Customer ID's are not uniformly distributed and might be strongly correlated with specific types of transactions like returns or system adjustments.*

### **Analyzing Transactions with Negative Quantities**

Negative quantities often represent returns or cancellations.

In [ ]:
# Filter the DataFrame for rows where Quantity is less than 0
negative_quantity_df = df[df['Quantity'] < 0]

print(f"Total number of transactions with negative quantities: {len(negative_quantity_df)}")

# Display the first few rows of these transactions
display(negative_quantity_df.head(10))

Total number of transactions with negative quantities: 12302


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom


Transactions with negative quantities usually indicate returns or cancelled orders.

I examined these transactions more closely and:

- Identified **12,302 transactions** with negative quantities.

- **Invoice Prefix 'C':** many invoices start with 'C' (e.g., C489449). This is a common convention in retail datasets to denote 'Credit' transactions, which are typically returns or cancellations. This aligns with our expectation for negative quantities.

- **Associated Customer IDs:** Unlike some of the transactions with missing 'Customer ID' that had positive quantities (suggesting guest checkouts), many of these negative quantity transactions do have associated Customer IDs (e.g., 16321.0). This indicates that they are likely returns from existing customers.

- **Price and Description:** These transactions also generally have valid Price and Description values, further supporting that they are reversals of legitimate sales.

*My observations confirm that negative quantities largely represent returns or cancellations, which is crucial for data cleaning and accurate sales analysis. So I kept them in the dataset.*

**Transactions with Negative Quantity and Missing Customer ID**

In [ ]:
# Filter for transactions with negative quantity and missing Customer ID
negative_quantity_missing_customer_id_df = df[(df['Quantity'] < 0) & (df['Customer ID'].isna())]

print(f"Number of negative quantity transactions with missing Customer IDs: {len(negative_quantity_missing_customer_id_df)}")

# Display the first few rows to inspect
display(negative_quantity_missing_customer_id_df.head(10))

Number of negative quantity transactions with missing Customer IDs: 2486


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom


In [ ]:
# Analyze 'Description' column for negative quantity transactions with missing Customer ID
description_analysis = negative_quantity_missing_customer_id_df['Description'].value_counts(dropna=False)

print("Top 20 Descriptions for negative quantity transactions with missing Customer IDs:")
display(description_analysis.head(20))

# Also explicitly check for missing descriptions in this subset
missing_descriptions_count = negative_quantity_missing_customer_id_df['Description'].isna().sum()
print(f"\nNumber of missing descriptions in this subset: {missing_descriptions_count}")

Top 20 Descriptions for negative quantity transactions with missing Customer IDs:


,count
Description,
NaN,1827
Manual,69
?,42
SAMPLES,40
damages,39
damaged,36
Bank Charges,33
missing,24
POSTAGE,19



Number of missing descriptions in this subset: 1827


The analysis of the 'Description' column for transactions with negative quantities and missing Customer IDs provides some telling insights:

- **High Number of Missing Descriptions:** A significant portion, **1,827** out of these transactions, also have a **missing 'Description' (NaN)**. This indicates that for many of these potentially problematic entries, we lack product-specific information.

- **Common Non-Product Descriptions:** The most frequent non-missing descriptions include terms like **'Manual', '?', 'SAMPLES', 'damages', 'damaged', 'Bank Charges', 'missing', 'POSTAGE', and 'AMAZON FEE'**. These descriptions strongly suggest that these entries are not typical product sales or returns. Instead, they likely represent:
   1. Manual Adjustments: 'Manual', '?'
   2. Internal Inventory Adjustments/Losses: 'SAMPLES', 'damages', 'damaged', 'missing', 'smashed', 'crushed', 'MIA'
   3. Fees/Charges: 'Bank Charges', 'POSTAGE', 'AMAZON FEE', 'ebay sales'

*This pattern further supports the idea that these transactions (negative quantity, missing Customer ID) are often non-standard operational adjustments or internal accounting entries, rather than direct customer-facing sales or returns of specific products.*

*I excluded these transactions from my analyses as it is focusing on customer behavior or product sales.*

In [ ]:
# Drop transactions where Quantity is negative AND Customer ID is missing
# We filter for rows NOT matching these criteria
df = df[~((df['Quantity'] < 0) & (df['Customer ID'].isna()))]

print("DataFrame shape after dropping transactions with negative quantity and missing Customer IDs:")
print(df.shape)

DataFrame shape after dropping transactions with negative quantity and missing Customer IDs:
(516110, 8)


### **Exploring Rows with Negative Quantities and Valid Customer IDs**

Next I nalyzed transactions where the `Quantity` is negative, indicating returns or cancellations, but a `Customer ID` is present. These are typically legitimate returns or cancellations from known customers.

In [ ]:
# Filter for transactions with negative quantity and a VALID Customer ID
negative_quantity_valid_customer_id_df = df[(df['Quantity'] < 0) & (df['Customer ID'].notna())]

print(f"Total number of transactions with negative quantities and valid Customer IDs: {len(negative_quantity_valid_customer_id_df)}")

# Display the first few rows to inspect
display(negative_quantity_valid_customer_id_df.head(10))

Total number of transactions with negative quantities and valid Customer IDs: 9816


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom


There are **9,816 transactions** that have negative quantities and valid Customer IDs. Examining the sample:

- **Invoice Prefix 'C':** Similar to our previous observation, many invoices here also start with 'C' (e.g., C489449). This consistently indicates 'Credit' transactions, which are typically returns or cancellations.

- **Associated Customer IDs:** All these transactions have a valid Customer ID (e.g., 16321.0, 17984.0), confirming that these are returns or cancellations made by known customers.

**Price and Description:** These transactions generally have valid Price and Description values, which further supports that they are reversals of legitimate sales of specific products.

*These observations reinforce that such entries are genuine customer returns or cancellations.*

*I will treat these as deductions from total sales to calculate net revenue, or analyze them separately to understand **return patterns**.*

### **Re-exploring Rows with Missing 'Customer ID'**

After previous data cleaning steps, I revisited transactions where the `Customer ID` is missing to understand their characteristics better.

In [ ]:
# Re-filter for rows with missing Customer ID from the updated DataFrame
missing_customer_id_df = df[df['Customer ID'].isna()]

print(f"Total number of rows with missing Customer ID (after cleaning): {len(missing_customer_id_df)}")

display(missing_customer_id_df.head())

# Analyze the distribution of Quantity in this subset
quantity_distribution = missing_customer_id_df['Quantity'].apply(lambda x: 'Positive' if x > 0 else 'Negative/Zero').value_counts()
print("\nQuantity distribution for transactions with missing Customer ID:")
display(quantity_distribution)

# Analyze 'Description' column for these transactions
description_analysis_missing_id = missing_customer_id_df['Description'].value_counts(dropna=False)
print("\nTop 20 Descriptions for transactions with missing Customer IDs:")
display(description_analysis_missing_id.head(20))

Total number of rows with missing Customer ID (after cleaning): 105347


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom



Quantity distribution for transactions with missing Customer ID:


,count
Quantity,
Positive,105347



Top 20 Descriptions for transactions with missing Customer IDs:


,count
Description,
NaN,1101
DOTCOM POSTAGE,733
REGENCY CAKESTAND 3 TIER,340
PARTY BUNTING,310
STRAWBERRY CERAMIC TRINKET BOX,307
WHITE HANGING HEART T-LIGHT HOLDER,303
PACK OF 72 RETRO SPOT CAKE CASES,252
ANTIQUE SILVER TEA GLASS ETCHED,238
RECYCLING BAG RETROSPOT,232


Re-examining rows with missing Customer IDs, I found:

**Total Count:** There are now **105,347 rows** with missing Customer IDs. This number is slightly lower than before I dropped the negative quantity transactions with missing customer IDs.

**Quantity Distribution:** Interestingly, all of these remaining 105,347 transactions have **positive quantities**. This indicates that the previous step of removing negative quantity transactions with missing Customer IDs was effective. The remaining missing Customer ID entries now predominantly represent sales where the customer was not identified.

**Top Descriptions:**

- A significant **1,101** of these entries still have a **missing 'Description' (NaN)**.

- Common descriptions include *'DOTCOM POSTAGE' (733 entries), and various product names like 'REGENCY CAKESTAND 3 TIER', 'PARTY BUNTING', and 'STRAWBERRY CERAMIC TRINKET BOX'*.

*This analysis suggests that the majority of these missing Customer ID entries are likely from legitimate sales (positive quantities), potentially representing **'guest checkouts'** or situations where customer details weren't recorded. However, the presence of 'DOTCOM POSTAGE' and other non-product descriptions indicates some **operational or service charges** without an associated customer.*

*I will exclude them from some specific customer-centric analyses.*

In [ ]:
# Drop transactions where Customer ID is missing AND Description is missing
# We filter for rows NOT matching these criteria
df = df[~((df['Customer ID'].isna()) & (df['Description'].isna()))]

print("DataFrame shape after dropping transactions with missing Customer ID and Description:")
print(df.shape)

# Re-filter for rows with missing Customer ID from the updated DataFrame
# These should now all have a Description (as we just dropped rows without one)
remaining_missing_customer_id_df_with_desc = df[df['Customer ID'].isna()]

print(f"\nNumber of remaining rows with missing Customer ID (now all with description): {len(remaining_missing_customer_id_df_with_desc)}")

# Analyze the Price distribution for these rows
print("\nPrice distribution for remaining transactions with missing Customer IDs (and valid Description):")
display(remaining_missing_customer_id_df_with_desc['Price'].describe())

DataFrame shape after dropping transactions with missing Customer ID and Description:
(515009, 8)

Number of remaining rows with missing Customer ID (now all with description): 104246

Price distribution for remaining transactions with missing Customer IDs (and valid Description):


,Price
count,104246.000000
mean,6.673243
std,275.418202
min,-53594.360000
25%,1.700000
50%,3.360000
75%,5.910000
max,25111.090000


### **Exploring Rows with Negative Prices**

Transactions where the `Price` is negative often represent refunds, cancellations, or corrections where money was repaid to the customer.

In [ ]:
# Filter the DataFrame for rows where Price is less than 0
negative_price_df = df[df['Price'] < 0]

print(f"Total number of transactions with negative prices: {len(negative_price_df)}")

# Display the first few rows of these transactions
display(negative_price_df.head())

Total number of transactions with negative prices: 3


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom


There are only **3 transactions** with negative prices. Upon inspection, these are quite distinct:

**Invoice Prefix 'A':** Notice the invoices start with 'A' (e.g., A506401). This is often used in retail datasets to denote **'Adjustments' or special internal invoices**, rather than regular sales or returns ('C' prefix).

**StockCode 'B' and Description 'Adjust bad debt':** All three entries have a StockCode of 'B' and the Description **'Adjust bad debt'**. This strongly indicates that these are not product-related transactions but rather financial write-offs or accounting adjustments, where a large negative amount is being recorded.

**Large Negative Prices:** The prices are significantly negative (e.g., -53594.36), consistent with substantial financial adjustments.

**Missing Customer ID:** All these transactions have a Customer ID of NaN. This reinforces the idea that these are internal adjustments, not linked to a specific customer's purchase or return activity.

*These transactions are clearly not typical sales or returns but rather internal financial adjustments related to bad debt. So I excluded them from my analyses.*

In [ ]:
# Drop transactions where Price is negative
print("DataFrame shape before dropping transactions with negative prices:")
print(df.shape)

# filter for only positive prices
df = df[df['Price'] >= 0]

print("DataFrame shape after dropping transactions with negative prices:")
print(df.shape)

DataFrame shape before dropping transactions with negative prices:
(515009, 8)
DataFrame shape after dropping transactions with negative prices:
(515006, 8)


### **Exploring Rows with Zero Prices**

Transactions where the `Price` is zero could signify free items, promotional samples, or potentially data entry errors.

In [ ]:
# Filter the DataFrame for rows where Price is equal to 0
zero_price_df = df[df['Price'] == 0]

print(f"Total number of transactions with zero prices: {len(zero_price_df)}")

# Display the first few rows of these transactions
display(zero_price_df.head(20))

Total number of transactions with zero prices: 459


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,NaN,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231.0,United Kingdom
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108.0,United Kingdom
31993,491971,85042,ANTIQUE LILY FAIRY LIGHTS,1,2009-12-14 18:37:00,0.0,NaN,United Kingdom
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.0,15070.0,United Kingdom
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.0,18071.0,United Kingdom
47126,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.0,14258.0,United Kingdom


I identified **459 transactions** with zero prices. Examining the sample rows:

**Missing Customer IDs:** Many of these transactions also have missing Customer IDs (e.g., rows 5904, 31993). This aligns with my earlier observation of system errors/adjustments where both customer information and a valid price might be absent.

**Specific Descriptions:** some description include 'DOTCOM POSTAGE' (e.g., row 5904) and 'this is a test product' (e.g., rows 89084, 89180). This strongly indicates non-product-related charges or adjustments, such as shipping fees that might have been waived or were internal entries.

**Positive Quantities:** All these transactions still have positive quantities which means they **are not 'returns'** in the typical sense.

**Valid Customer ID with Zero Price:** Interestingly, some zero-price transactions do have a valid Customer ID (e.g., row 4674). Some are also clearly indicate as test products (e.g., 'this is a test product' in rows 89084, 89180). These could potentially be free samples, promotional items, or items included as part of a larger purchase without an individual cost.

*Overall, transactions with zero prices appear to be a mix of internal adjustments, possibly free giveaways, or promotional items, and are often not standard product sales.*

*I excluded them from my analytics*.

In [ ]:
# Drop rows where Price is 0
df = df[df['Price'] != 0]

print("DataFrame shape after dropping transactions with zero prices:")
print(df.shape)

DataFrame shape after dropping transactions with zero prices:
(514547, 8)


### **Final Check for Missing Values**

In [ ]:
# Calculate missing values for the entire DataFrame
final_missing_values = df.isna().sum()
final_percentage_missing = (final_missing_values / len(df)) * 100

final_missing_info = pd.DataFrame({
    'Missing Count': final_missing_values,
    'Percentage Missing': final_percentage_missing.round(2)
})

print("Final missing values and their percentages across all columns:")
display(final_missing_info[final_missing_info['Missing Count'] > 0].sort_values(by='Percentage Missing', ascending=False))

Final missing values and their percentages across all columns:


,Missing Count,Percentage Missing
Customer ID,103815,20.18


# Creating TransactionType Column

In most retail sales datasets, distinguishing between sales and returns/cancellations is a fundamental step that significantly enhances the clarity and accuracy of the analysis.

Because I identified transactions with negative quantities that indicate **returns/cancellationg** in my previous cleaning steps, I am creating a new column named `TansactionType` that will be used to explicitly distinguish sales from retun/cancelled transactions.

This step will be useful later for:
- Sales Analysis such as *sales trends, product performance, or customer purchasing behavior* and
- Return/Cancellation Analysis to understand return patterns (e.g., which products are returned most often, who are the high-return customers, reasons for cancellation)
- Net Sales Calculation for accurate financial reporting or analysis of actual revenue
- Foundation for ML: This distinction will almost certainly be a crucial feature or even a target variable in Machine Learning (e.g., predicting customer churn, return probability, or classifying transaction types).

*Transactions with quantity less than 0 are labeled as 'Return/Cancelled' while those with quantity greather than 0 are labeled 'Sale'*

*It explicitly defines the nature of each row, which is a fundamental piece of information for any business analysis.*

In [ ]:
# Create 'TransactionType' column
df['TransactionType'] = df['Quantity'].apply(lambda x: 'Return/Cancelled' if x < 0 else 'Sale')

print("DataFrame with new 'TransactionType' column:")
display(df.head())

DataFrame with new 'TransactionType' column:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TransactionType
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Sale
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Sale
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Sale
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Sale
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Sale


In [ ]:
# Save the cleaned DataFrame to a CSV file
output_path = '/content/drive/My Drive/cleaned_retail_data.csv'

# Save the DataFrame to the specified path in Google Drive
df.to_csv(output_path, index=False)

print(f"Cleaned DataFrame saved to: {output_path}")

Cleaned DataFrame saved to: /content/drive/My Drive/cleaned_retail_data.csv
